In [1]:
import ee
import matplotlib.pyplot as plt

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Define location: Veracruz, Mexico
point = ee.Geometry.Point([-96.870102, 19.777411]) #rancho, villanueva, ver
#point = ee.Geometry.Point([-96.873012,19.705823]) #miahuatlan, ver

# Time range
start_date = '2025-05-01'
end_date = '2025-06-30'

# Load CHIRPS daily rainfall
chirps = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterBounds(point)
    .filterDate(start_date, end_date)
)

print("Image count:", chirps.size().getInfo())
test_image = chirps.first()
test_value = test_image.reduceRegion(
    ee.Reducer.mean(), point, scale=5000
).getInfo()

print("Sample value from first image:", test_value)


def extract_precip(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    value = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=2000
    ).get('precipitation')
    return ee.Feature(None, {
        'date': date,
        'precip': value
    })

# First map all features
features_all = chirps.map(extract_precip)
#print("features_all: ", features_all)

# Then filter out nulls
features_valid = features_all.filter(ee.Filter.notNull(['precip']))
print("Valid features count:", features_valid.size().getInfo())



# Download properties
dates = features_valid.aggregate_array('date').getInfo()
values = features_valid.aggregate_array('precip').getInfo()
#print("records ",records)


# Map the extraction over the collection
#features = chirps.map(extract_precip).filter(ee.Filter.notNull(['precip']))

# Get as a list of dicts
#records = features.aggregate_array('properties').getInfo()

# Convert to Python lists
#dates = [r['date'] for r in records]
#values = [r['precip'] for r in records]

print ("values = ", values)

# Plot in-memory
plt.figure(figsize=(12, 5))
plt.plot(dates, values, marker='o', linestyle='-')
plt.xticks(rotation=45)
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.title('Daily Rainfall (CHIRPS) at Veracruz (2023)')
plt.grid(True)
plt.tight_layout()
plt.show()


Enter verification code:  4/1AVGzR1BKZelmHA4mLB3zqnd16UPjM4pbd7HnM7GM4tngdTvwW2VMlNL0ejw



Successfully saved authorization token.


EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.